# Double ML causal inference

### Импорты

In [1]:
import pandas as pd
import doubleml as dml
from sklearn.ensemble import RandomForestClassifier

alpha = 0.1

### Данные

In [2]:
df = pd.read_csv("data/final_dataset.csv")
df_men = df[df["sex"] == 1]
df_women = df[df["sex"] == 2]

In [3]:
df_men

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
8,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
16,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4586,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
4587,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0
4589,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4592,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


In [4]:
df_women

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4593,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4594,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4596,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


### Женщины и гипертония

In [5]:
treatment = "diploma"
outcome = "hypertension"
controls = ["age", "invalid", "type_area", "income"]

In [6]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [7]:
ml_g = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)
ml_m = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)

dml_irm = dml.DoubleMLIRM(
    dml_data_women,
    ml_g=ml_g,
    ml_m=ml_m,
    trimming_threshold=0.01,
    n_folds=5,
    score="ATE",
)

dml_irm.fit()

print(dml_irm.summary)
print()
print(dml_irm.summary)

             coef   std err         t     P>|t|     2.5 %    97.5 %
diploma -0.035574  0.016198 -2.196237  0.028075 -0.067321 -0.003827

             coef   std err         t     P>|t|     2.5 %    97.5 %
diploma -0.035574  0.016198 -2.196237  0.028075 -0.067321 -0.003827


### Женщины и заболевания глаз

In [8]:
treatment = "diploma"
outcome = "eyes"
controls = ["age", "invalid", "type_area", "income"]

In [9]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [10]:
ml_g = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)
ml_m = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)

dml_irm = dml.DoubleMLIRM(
    dml_data_women,
    ml_g=ml_g,
    ml_m=ml_m,
    trimming_threshold=0.01,
    n_folds=5,
    score="ATE",
)

dml_irm.fit()

print(dml_irm.summary)
print()
print(dml_irm.summary)

             coef   std err         t     P>|t|    2.5 %    97.5 %
diploma  0.013697  0.013632  1.004814  0.314986 -0.01302  0.040415

             coef   std err         t     P>|t|    2.5 %    97.5 %
diploma  0.013697  0.013632  1.004814  0.314986 -0.01302  0.040415


### Женщины и аллергия

In [11]:
treatment = "diploma"
outcome = "allergy"
controls = ["age", "invalid", "type_area", "income"]

In [12]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [13]:
ml_g = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)
ml_m = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)

dml_irm = dml.DoubleMLIRM(
    dml_data_women,
    ml_g=ml_g,
    ml_m=ml_m,
    trimming_threshold=0.01,
    n_folds=5,
    score="ATE",
)

dml_irm.fit()

print(dml_irm.summary)
print()
print(dml_irm.summary)

             coef   std err         t     P>|t|     2.5 %    97.5 %
diploma -0.001821  0.011296 -0.161223  0.871918 -0.023961  0.020319

             coef   std err         t     P>|t|     2.5 %    97.5 %
diploma -0.001821  0.011296 -0.161223  0.871918 -0.023961  0.020319


### Мужчины и оценка состояния здоровья

In [14]:
treatment = "diploma"
outcome = "is_health_very_good"
controls = ["age", "invalid", "type_area", "income"]

In [15]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [16]:
ml_g = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)
ml_m = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)

dml_irm = dml.DoubleMLIRM(
    dml_data_men,
    ml_g=ml_g,
    ml_m=ml_m,
    trimming_threshold=0.01,
    n_folds=5,
    score="ATE",
)

dml_irm.fit()

print(dml_irm.summary)
print()
print(dml_irm.confint(level=1 - alpha))

             coef   std err         t     P>|t|     2.5 %    97.5 %
diploma -0.000157  0.005566 -0.028294  0.977428 -0.011067  0.010752

            5.0 %    95.0 %
diploma -0.009313  0.008998


### Другие интересные наблюдения

In [17]:
treatment = "diploma"
controls = ["age", "invalid", "type_area", "income"]

diseases = [
    "heart",
    "lungs",
    "liver",
    "kidneys",
    "stomach",
    "spine",
    "diabetes",
    "hypertension",
    "joints",
    "ENT_organs",
    "neurology",
    "eyes",
    "allergy",
    "veins",
    "skin",
    "oncology",
    "is_health_good",
    "is_health_very_good",
]

ml_g = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)
ml_m = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
)

for outcome in diseases:
    dml_irm_women = dml.DoubleMLIRM(
        dml_data_women,
        ml_g=ml_g,
        ml_m=ml_m,
        trimming_threshold=0.01,
        n_folds=5,
        score="ATE",
    )
    dml_irm_women.fit()

    p_val_w = dml_irm_women.pval.item()

    if p_val_w <= alpha:
        print(f"Women - {outcome}:")
        print(dml_irm_women.summary)
        print()
        print(dml_irm_women.confint(level=1 - alpha))
    else:
        print(f"Women - {outcome}: no effect")
    print(3 * "\n")

    dml_irm_men = dml.DoubleMLIRM(
        dml_data_men,
        ml_g=ml_g,
        ml_m=ml_m,
        trimming_threshold=0.01,
        n_folds=5,
        score="ATE",
    )
    dml_irm_men.fit()

    p_val_m = dml_irm_men.pval.item()

    if p_val_m <= alpha:
        print(f"Men - {outcome}:")
        print(dml_irm_men.summary)
        print()
        print(dml_irm_men.confint(level=1 - alpha))
    else:
        print(f"Men - {outcome}: no effect")
    print(3 * "\n")

Women - heart: no effect




Men - heart: no effect




Women - lungs: no effect




Men - lungs: no effect




Women - liver: no effect




Men - liver: no effect




Women - kidneys: no effect




Men - kidneys: no effect




Women - stomach: no effect




Men - stomach: no effect




Women - spine: no effect




Men - spine: no effect




Women - diabetes: no effect




Men - diabetes: no effect




Women - hypertension: no effect




Men - hypertension: no effect




Women - joints: no effect




Men - joints: no effect




Women - ENT_organs: no effect




Men - ENT_organs: no effect




Women - neurology: no effect




Men - neurology: no effect




Women - eyes: no effect




Men - eyes: no effect




Women - allergy: no effect




Men - allergy: no effect




Women - veins: no effect




Men - veins: no effect




Women - skin: no effect




Men - skin: no effect




Women - oncology: no effect




Men - oncology: no effect




Women - is_health_good: no effect




Men - is_hea

### Выводы